In [23]:
from liteparse import LiteParse
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_chroma import Chroma

In [8]:
load_dotenv()

True

In [2]:
PDF_PATH = Path("../data/seq2seq.pdf").resolve()

In [3]:
parser = LiteParse()
content = parser.parse(PDF_PATH)

[liteparse] extract: 171.1ms (9 pages)
[liteparse] ocr render: 10.8ms (1 pages)
[liteparse] ocr: 2667.2ms
[liteparse] project: 2.0ms
[liteparse] total: 2851.4ms


In [4]:
sanitized_text = content.text.strip().replace("\n", " ")
raw_documents = [Document(page_content=sanitized_text)]

In [5]:
textsplitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, length_function=len, is_separator_regex=False)
documents = textsplitter.split_documents(raw_documents)

In [21]:
def add_metadata_to_documents(documents):
    for idx, doc in enumerate(documents):
        doc.metadata = {"source": doc.metadata.get("source", "https://arxiv.org/abs/1409.3215")}
        doc.id = idx
    return documents

documents = add_metadata_to_documents(documents)

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512)

In [24]:
vector_store = Chroma(
    collection_name="research_paper_documents",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",
)

In [63]:
document1 = Document(page_content="Hi my name is krish", metadata={"source": "profile.txt"})
document2 = Document(page_content="Krish go to school at UCB", metadata={"source": "studies.txt"})

In [43]:
vector_store.add_documents([document1])

['a7a5c043-fbe1-4cdd-89b9-f5092dfa9e3f']

In [48]:
vector_store.get()

{'ids': ['a7a5c043-fbe1-4cdd-89b9-f5092dfa9e3f'],
 'embeddings': None,
 'documents': ['Hi my name is krish'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'profile.txt'}]}

In [49]:
document1.page_content = "Hi my name is krish from future world"

In [51]:
vector_store.update_document(
    document_id="a7a5c043-fbe1-4cdd-89b9-f5092dfa9e3f",
    document=document1
)

In [52]:
vector_store.get()

{'ids': ['a7a5c043-fbe1-4cdd-89b9-f5092dfa9e3f'],
 'embeddings': None,
 'documents': ['Hi my name is krish from future world'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'profile.txt'}]}

In [53]:
vector_store.add_documents([document2])

['227aa6fb-c076-4c91-ae7c-ab9db34e8726']

In [54]:
vector_store.get()

{'ids': ['a7a5c043-fbe1-4cdd-89b9-f5092dfa9e3f',
  '227aa6fb-c076-4c91-ae7c-ab9db34e8726'],
 'embeddings': None,
 'documents': ['Hi my name is krish from future world',
  'I go to school at UCB'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'profile.txt'}, {'source': 'studies.txt'}]}

In [61]:
vector_store.similarity_search("Where does krish go to school?", k=1)

[Document(id='227aa6fb-c076-4c91-ae7c-ab9db34e8726', metadata={'source': 'studies.txt'}, page_content='Krish go to school at UCB')]

In [62]:
retriever = vector_store.as_retriever(search_kwargs={"k": 1})
retriever.invoke("Where does krish go to school?")

[Document(id='227aa6fb-c076-4c91-ae7c-ab9db34e8726', metadata={'source': 'studies.txt'}, page_content='Krish go to school at UCB')]

Sequence to Sequence

In [67]:
pdf_store = Chroma(
    collection_name="arxiv_paper_documents",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",
)

In [70]:
pdf_store.add_documents(documents, ids=[str(i) for i in range(len(documents))])

['0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 '10',
 '11',
 '12',
 '13',
 '14',
 '15',
 '16',
 '17',
 '18',
 '19',
 '20',
 '21',
 '22',
 '23',
 '24',
 '25',
 '26',
 '27',
 '28',
 '29',
 '30',
 '31',
 '32',
 '33',
 '34',
 '35',
 '36',
 '37',
 '38',
 '39',
 '40',
 '41',
 '42',
 '43',
 '44',
 '45']

In [ ]:
retriever = pdf_store.as_retriever(search_type="similarity_score_threshold", search_kwargs={"k": 1})

In [86]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful assistant that can answer questions about the paper.
    Here is the paper:
    {context}
    Question: {question}
    Answer:
    """
)

def ask_rag(question):
    context = retriever.invoke(question)
    response = llm.invoke(prompt.format(context=context, question=question))
    return response.content

In [87]:
response = ask_rag("What is the main author of this paper?")

In [88]:
response

'The main author of the paper is not explicitly mentioned in the provided text. However, the references included suggest that it may involve contributions from notable researchers in the field of neural networks, such as S. Hochreiter and J. Schmidhuber, who are known for their work on Long Short-Term Memory (LSTM) networks. If you have more specific information or context about the paper, I can help identify the main author.'

In [89]:
pdf_store.similarity_search("What is the main author of this paper?", k=1)

[Document(id='42', metadata={'source': 'https://arxiv.org/abs/1409.3215'}, page_content="neural networks. In Arxiv preprint arXiv:1308.0850, 2013.   [11] A. Graves, S. Fern´andez, F. Gomez, and J. Schmidhuber. Connectionist temporal classification: labelling unsegmented sequence data with recurrent neural networks. In ICML, 2006.          [12] K. M. Hermann and P. Blunsom. Multilingual distributed representations without word alignment. In ICLR, 2014.             [13] G. Hinton, L. Deng, D. Yu, G. Dahl, A. Mohamed, N. Jaitly, A. Senior, V. Vanhoucke, P. Nguyen, T. Sainath, and B. Kingsbury. Deep neural networks for acoustic modeling in speech recognition. IEEE Signal Processing Magazine, 2012.       [14] S. Hochreiter. Untersuchungen zu dynamischen neuronalen netzen. Master's thesis, Institut fur Infor- matik, Technische Universitat, Munchen, 1991. [15] S. Hochreiter, Y. Bengio, P. Frasconi, and J. Schmidhuber. Gradient flow in recurrent nets: the difficulty of learning long-term depen